In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

In [ ]:
def extract_text(filepath:str) -> str:
    file_path=Path(filepath)
    text = ""
    if not file_path.exists():
        raise FileNotFoundError(f"File not fround:{file_path}")
    if file_path.suffix.lower()==".pdf":
        import fitz   
        doc = fitz.open(file_path)
        text  = ""
        for page in doc:
            pagetext=page.get_text("text")
            if isinstance(pagetext,str):
                text+=pagetext
            elif isinstance(pagetext,list):
                text+="".join(map(str,pagetext))
            elif isinstance(pagetext,dict):
                text+=str(pagetext)    
        doc.close()
        return text    
    elif file_path.suffix.lower() == ".docx":
        from docx import Document
        doc =Document(filepath)
        return "n".join(paragraph.text for paragraph in doc.paragraphs)
    else:
        raise ValueError("not good file system")






    return text


In [ ]:
cringe = extract_text("Reseach paper Summary-1.pdf")

In [ ]:
import google.generativeai as genai

In [ ]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
def generate_gemini_mindmap(text: str, model_name: str = "models/gemini-2.5-flash") -> str:
    if not text or not text.strip():
        raise ValueError("Input text is empty. Please provide a non-empty document.")

    genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
    model = genai.GenerativeModel(model_name)

    prompt = f"""Create a hierarchical mind map of the following document.
Use a nested bullet list with dashes (-) and 2-space indentation per level.
Limit depth to 5 levels. Only output the mind map.

Document:
{text}
"""

    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            max_output_tokens=1048576 ,
            temperature=0.3,
        ),
        safety_settings=[
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
        ],
    )

    # Extract text manually
    output = []
    if getattr(response, "candidates", None):
        for c in response.candidates:
            if getattr(c, "content", None) and getattr(c.content, "parts", None):
                for p in c.content.parts:
                    if hasattr(p, "text"):
                        output.append(p.text)

    if output:
        return "\n".join(output).strip()

    finish_reasons = [getattr(c, "finish_reason", None) for c in getattr(response, "candidates", [])]
    raise RuntimeError(
        f"No text returned. Finish reasons: {finish_reasons}. "
        f"The model may have blocked output or received empty/unsafe content."
    )


In [ ]:
CRINGE =generate_gemini_mindmap(cringe)

In [ ]:
def parse_bullet_mindmap(bullet_text: str):
    """
    Parse a nested bullet list (with 2-space indentation per level)
    into a tree structure: list of dicts with 'title' and 'children'.
    """
    lines = bullet_text.strip().split('\n')
    stack = []  
    root = []

    for line in lines:
        if not line.strip():
            continue

        raw_line = line
        content = raw_line.lstrip(' ')
        if not content.startswith('- '):
            continue  
        content = content[2:].strip() 

        leading_spaces = len(raw_line) - len(raw_line.lstrip(' '))
        depth = leading_spaces // 2  
        node = {'title': content, 'children': []}

        while stack and stack[-1][1] >= depth:
            stack.pop()

        if depth == 0:
            root.append(node)
            stack = [(node, depth)]
        else:
            if stack:
                parent = stack[-1][0]
                parent['children'].append(node)
            stack.append((node, depth))

    return root


In [ ]:
ass =parse_bullet_mindmap(CRINGE)

In [ ]:
ass

In [ ]:
def tree_to_pyvis(tree_data, title="Mind Map", height="1000px", width="100%"):
    """
    Convert a nested tree (list of dicts with 'title' and 'children') to a Pyvis network.
    Returns a Network object ready to be saved or rendered.
    """
    net = Network(
        height=height,
        width=width,
        bgcolor="#ffffff",
        font_color="black",
        directed=True
    )

    # Optional: customize styling
    net.set_options("""
    {
      "nodes": {
        "shape": "box",
        "font": {"size": 14, "face": "Arial"},
        "margin": 10,
        "widthConstraint": {"maximum": 300}
      },
      "edges": {
        "arrows": {"to": {"enabled": true}},
        "smooth": {"type": "cubicBezier"}
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 0
      },
      "physics": {
        "enabled": true,
        "barnesHut": {
          "gravitationalConstant": -8000,
          "centralGravity": 0.3,
          "springLength": 95,
          "springConstant": 0.04
        }
      }
    }
    """)

    node_id_counter = [0]  

    def add_node(parent_id, node_dict, depth=0):
        # Generate unique ID
        node_id = f"node_{node_id_counter[0]}"
        node_id_counter[0] += 1

        # Truncate long titles for display
        display_title = node_dict['title'][:60] + "..." if len(node_dict['title']) > 60 else node_dict['title']

        # Color by depth
        color = "#d5e8d4" if depth == 0 else "#f8cecc" if depth == 1 else "#e1d5e7"

        net.add_node(
            node_id,
            label=display_title,
            title=node_dict['title'],  
            color=color,
            size=20 + depth * 5
        )

        if parent_id:
            net.add_edge(parent_id, node_id)

        # Recurse into children
        for child in node_dict.get('children', []):
            add_node(node_id, child, depth + 1)

    for root in tree_data:  
        add_node(None, root, 0)

    return net

In [ ]:
from pyvis.network import Network
sus =tree_to_pyvis(ass)

NameError: name 'ass' is not defined

In [ ]:
sus.save_graph("mindmap2.html")

## USING HDBSCAN

In [ ]:
def generate_clustering_mindmap(text: str, min_cluster_size: int = 3):
    """
    Generate a mind map using local embeddings + UMAP + HDBSCAN.
    Returns a tree in the same format as parse_bullet_mindmap().
    """
    import re
    from sentence_transformers import SentenceTransformer
    import umap
    import hdbscan
    import numpy as np

    # --- 1. Split text into meaningful chunks ---
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if 20 < len(s.strip()) < 500]

    if len(sentences) < 5:
        sentences = [p.strip() for p in text.split('\n') if len(p.strip()) > 30]
    if len(sentences) < 2:
        sentences = [text[i:i+300] for i in range(0, len(text), 300)][:20]

    if not sentences:
        return [{"title": "Clustering failed: no valid text", "children": []}]

    # --- 2. Embed sentences ---
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(sentences, show_progress_bar=False)

    # --- 3. Reduce dimensionality ---
    umap_model = umap.UMAP(n_neighbors=5, n_components=5, random_state=42)
    umap_embeds = umap_model.fit_transform(embeddings)

    # --- 4. Cluster ---
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=1,
        metric='euclidean',
        cluster_selection_method='leaf'
    )
    clusters = clusterer.fit_predict(umap_embeds)

    # --- 5. Build tree ---
    unique_clusters = sorted(set(clusters))
    root_nodes = []

    for cluster_id in unique_clusters:
        if cluster_id == -1:  # noise
            continue

        members = [sentences[i] for i, c in enumerate(clusters) if c == cluster_id]
        if not members:
            continue

        # Create label from first member
        label = " ".join(members[0].split()[:6])
        if len(members[0].split()) > 6:
            label += "..."

        children = []
        for m in members[:4]:  # max 4 per cluster
            short = (m[:70] + "...") if len(m) > 70 else m
            children.append({"title": short, "children": []})

        root_nodes.append({
            "title": f"Topic {cluster_id}: {label}",
            "children": children
        })

    if not root_nodes:
        return [{"title": "No clusters found (try smaller min_cluster_size)", "children": []}]

    return root_nodes

In [ ]:
# Generate clustering-based mind map
clustering_tree = generate_clustering_mindmap(cringe, min_cluster_size=3)

c:\ita-mini\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\ita-mini\venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\ita-mini\venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
clustering_net = tree_to_pyvis(clustering_tree)
clustering_net.save_graph("clustering_mindmap.html")
print("✅ Clustering mind map saved as 'clustering_mindmap.html'")

✅ Clustering mind map saved as 'clustering_mindmap.html'
